# Estudio 06 · Fase 1a · Más historia (solo epidemiología)

Comparación 1a, preregistrada en `code/preregistration.py → PHASE1` (commit `6a29a8e`; enmienda G1 en `81e72a4`):

- **L:** C3 (`base_6` + B3) · E3 · S4_s3 con los hiperparámetros de la V1, entrenada desde 2005; bloques por temporada
  2008–2024 y `calendario_2025` (2008 solo calibra).
- **R:** la misma configuración entrenada desde 2017 con la tabla de orígenes exacta de la V1; bloques 2018–2024 y
  `calendario_2025` (2018 solo calibra).
- Platt de cada bloque con todas las temporadas cerradas anteriores; alerta p ≥ 0,50; 5 semillas.
- **Deciden 2022 y 2023** (R no tiene entrenamiento antes de 2017); 2024 es contraste. Se adopta L si gana o empata.

Nada se escribe en `data/`; las predicciones van a `models/experimentos/mejora_modelo/fase1a/` (no versionado).

In [1]:
import json
import sys

import pandas as pd

from denguekay.config.paths import RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.evaluation import protocol as pa
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase1_history as p1
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "fase1a.json"
device = xgboost_settings()
inp = p1.phase1a_inputs()
device

{'device': 'cuda', 'n_jobs': 12}

## 1 · Guardas antes de mirar resultados (G1 enmendada, G2, G3; G4 es un test)

In [2]:
g = p1.guards(inp, pr.G1_EXCEPTION)
failed = [k for k, v in g.items() if not v["pasa"]]
if failed:
    raise RuntimeError(f"Fallan las guardas {failed}: se detiene la fase. {g}")
g

{'G1': {'origenes_v1': 30160,
  'sin_par_en_la_larga': 0,
  'filas_distintas': {'hist_lag_a': 0,
   'hist_lag_b': 0,
   'hist_media_4': 0,
   'hist_semanas_pos_4': 0,
   'semana_epi_seno': 0,
   'semana_epi_coseno': 0,
   'propia_semanas_desde_ultimo_caso': 0,
   'propia_pendiente_4': 0,
   'propia_log_casos_temporada_anterior': 0},
  'diferencia_maxima': {'hist_lag_a': 0.0,
   'hist_lag_b': 0.0,
   'hist_media_4': 0.0,
   'hist_semanas_pos_4': 0.0,
   'semana_epi_seno': 0.0,
   'semana_epi_coseno': 0.0,
   'propia_semanas_desde_ultimo_caso': 43.0,
   'propia_pendiente_4': 0.0,
   'propia_log_casos_temporada_anterior': 0.0},
  'excepcion_G1': {'filas': 134,
   'distritos': 20,
   'coinciden_con_el_recalculo': 134},
  'pasa': np.True_},
 'G2': {'filas': 50895,
  'filas_guardadas': 50895,
  'cruzadas': 50895,
  'diferencia_maxima': {'casos_predichos': 0.0, 'probabilidad_brote': 0.0},
  'pasa': True},
 'G3': {'filas': 30485, 'filas_gold': 30485, 'distintas': 0, 'pasa': True}}

## 2 · Entrenamiento y evaluación (5 semillas)

In [3]:
L = p1.run_variant(inp["matrices_L"], inp["origenes_L"], p1.seasonal_blocks(p1.FIRST_BLOCK_SEASON_LONG),
                   inp["serie"], pr.SEEDS)
R = p1.run_variant(inp["matrices_R"], inp["origenes_R"], p1.seasonal_blocks(p1.FIRST_BLOCK_SEASON_2017),
                   inp["serie"], pr.SEEDS)
saved_v1 = p1.read_saved(p1.V1_SAVED)
V1 = {"evaluaciones": {s: pa.evaluate(g_.drop(columns="semilla"), inp["serie"])
                       for s, g_ in saved_v1.groupby("semilla")}}
p1.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, v in (("L", L), ("R", R)):
    pd.concat(v["predicciones"].values(), ignore_index=True).to_csv(
        p1.OUTPUT_DIR / f"predicciones_{name}.csv.gz", index=False)
    v["alfas"].to_csv(p1.OUTPUT_DIR / f"alfas_{name}.csv", index=False)

## 3 · Veredicto preregistrado (2022 y 2023)

In [4]:
dec = p1.decision_1a(L, R)
dec["veredicto"]

,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,gana,True,False,None,"brier, razon_mae",2,0,2,2,0,1,0,0,2,0
1,3,gana,True,False,None,"brier, razon_mae",2,1,2,2,0,1,0,0,2,0
2,4,gana,True,False,None,"brier, razon_mae",2,1,2,2,0,1,0,0,2,0


In [5]:
dec["antelacion"]

,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2022,diferencia_temporada_2023,episodios_temporada_2022,episodios_temporada_2023
0,2,95,0.147368,0.084211,0.221053,0.131579,0.157895,38,57
1,3,95,0.252632,0.168421,0.336842,0.184211,0.298246,38,57
2,4,95,0.273684,0.189474,0.357895,0.157895,0.350877,38,57


In [6]:
dec["metricas_L_R"].round(3)

,horizonte,bloque,positivos_L,f1_050_L,f1_050_R,recall_050_L,recall_050_R,precision_050_L,precision_050_R,auprc_L,auprc_R,brier_L,brier_R,razon_mae_L,razon_mae_R,razon_rmse_L,razon_rmse_R
5,2,temporada_2022,431.0,0.712,0.154,0.696,0.084,0.728,0.973,0.781,0.789,0.054,0.084,0.962,1.001,1.010,1.106
6,2,temporada_2023,1039.0,0.857,0.778,0.828,0.647,0.889,0.975,0.947,0.947,0.062,0.082,0.847,0.881,0.940,0.987
7,2,temporada_2024,857.0,0.799,0.800,0.853,0.777,0.752,0.824,0.893,0.891,0.074,0.072,1.259,1.352,1.566,1.673
13,3,temporada_2022,431.0,0.704,0.000,0.682,0.000,0.728,NaN,0.771,0.781,0.056,0.102,0.959,1.021,1.063,1.232
14,3,temporada_2023,1039.0,0.835,0.578,0.794,0.408,0.880,0.993,0.930,0.928,0.073,0.120,0.791,0.808,0.888,0.913
15,3,temporada_2024,857.0,0.788,0.766,0.863,0.709,0.725,0.830,0.876,0.870,0.079,0.081,1.382,1.515,1.778,1.890
21,4,temporada_2022,431.0,0.652,0.000,0.599,0.000,0.718,NaN,0.743,0.753,0.060,0.110,0.952,1.040,1.114,1.380
22,4,temporada_2023,1039.0,0.811,0.216,0.736,0.121,0.901,1.000,0.922,0.917,0.081,0.159,0.748,0.764,0.839,0.852
23,4,temporada_2024,857.0,0.761,0.695,0.824,0.590,0.706,0.842,0.840,0.831,0.089,0.095,1.471,1.622,1.892,2.014


In [7]:
print("Se adopta L como base de las fases siguientes:" if dec["adopta_L"] else
      "L compensa o pierde en algún horizonte: se detiene el estudio y se consulta a Rosa.", dec["adopta_L"])

Se adopta L como base de las fases siguientes: True


## 4 · Lecturas que no deciden

In [8]:
rd = p1.readings_1a(L, R, V1, inp["serie"], dec["medianas_L"])
rd["metricas_por_temporada_L"][["horizonte", "bloque", "positivos", "f1_050", "f1_050_rango", "recall_050",
    "precision_050", "auprc", "brier", "brier_sin_calibrar", "razon_mae", "razon_rmse"]].round(3)

,horizonte,bloque,positivos,f1_050,f1_050_rango,recall_050,precision_050,auprc,brier,brier_sin_calibrar,razon_mae,razon_rmse
0,2,temporada_2010,234.0,0.722,0.016,0.701,0.745,0.780,0.030,0.033,0.938,0.920
1,2,temporada_2015,557.0,0.772,0.008,0.695,0.865,0.885,0.051,0.065,0.945,1.053
2,2,temporada_2016,395.0,0.657,0.003,0.714,0.609,0.757,0.059,0.055,1.100,1.071
3,2,temporada_2017,539.0,0.703,0.006,0.744,0.664,0.800,0.068,0.069,0.800,0.875
4,2,temporada_2022,431.0,0.712,0.003,0.696,0.728,0.781,0.054,0.055,0.962,1.010
5,2,temporada_2023,1039.0,0.857,0.004,0.828,0.889,0.947,0.062,0.071,0.847,0.940
6,2,temporada_2024,857.0,0.799,0.002,0.853,0.752,0.893,0.074,0.074,1.259,1.566
7,3,temporada_2010,234.0,0.631,0.012,0.551,0.732,0.706,0.036,0.039,0.897,0.891
8,3,temporada_2015,557.0,0.649,0.008,0.515,0.875,0.839,0.064,0.081,0.895,1.045
9,3,temporada_2016,395.0,0.627,0.003,0.658,0.599,0.709,0.065,0.063,1.131,1.142


In [9]:
rd["hu0007_4_L"].round(3)

,horizonte,bloque,precision_050,recall_050,f1_050,razon_mae,razon_rmse,cumple_precision_050,cumple_recall_050,cumple_f1_050,cumple_razon_mae,cumple_razon_rmse,cumple_clasificacion,cumple_regresion,positivos,evaluable
0,2,temporada_2010,0.745,0.701,0.722,0.938,0.920,True,False,True,False,False,False,False,234.0,True
1,2,temporada_2015,0.865,0.695,0.772,0.945,1.053,True,False,True,False,False,False,False,557.0,True
2,2,temporada_2016,0.609,0.714,0.657,1.100,1.071,True,False,False,False,False,False,False,395.0,True
3,2,temporada_2017,0.664,0.744,0.703,0.800,0.875,True,False,True,True,False,False,False,539.0,True
4,2,temporada_2022,0.728,0.696,0.712,0.962,1.010,True,False,True,False,False,False,False,431.0,True
5,2,temporada_2023,0.889,0.828,0.857,0.847,0.940,True,True,True,True,False,True,False,1039.0,True
6,2,temporada_2024,0.752,0.853,0.799,1.259,1.566,True,True,True,False,False,True,False,857.0,True
7,3,temporada_2010,0.732,0.551,0.631,0.897,0.891,True,False,False,False,False,False,False,234.0,True
8,3,temporada_2015,0.875,0.515,0.649,0.895,1.045,True,False,False,False,False,False,False,557.0,True
9,3,temporada_2016,0.599,0.658,0.627,1.131,1.142,False,False,False,False,False,False,False,395.0,True


In [10]:
rd["calibracion_V1_frente_a_R"].round(3)

,horizonte,bloque,positivos,f1_050_V1,recall_050_V1,brier_V1,brier_sin_calibrar_V1,razon_mae_V1,f1_050_R,recall_050_R,brier_R,brier_sin_calibrar_R,razon_mae_R
2,2,temporada_2022,431.0,0.711,0.696,0.053,0.052,1.001,0.154,0.084,0.084,0.052,1.001
3,2,temporada_2023,1039.0,0.852,0.812,0.062,0.065,0.881,0.778,0.647,0.082,0.065,0.881
4,2,temporada_2024,857.0,0.796,0.866,0.076,0.072,1.352,0.800,0.777,0.072,0.072,1.352
7,3,temporada_2022,431.0,0.658,0.559,0.057,0.055,1.021,0.000,0.000,0.102,0.055,1.021
8,3,temporada_2023,1039.0,0.828,0.774,0.075,0.080,0.808,0.578,0.408,0.120,0.080,0.808
9,3,temporada_2024,857.0,0.778,0.873,0.086,0.079,1.515,0.766,0.709,0.081,0.079,1.515
12,4,temporada_2022,431.0,0.546,0.399,0.062,0.060,1.040,0.000,0.000,0.110,0.060,1.040
13,4,temporada_2023,1039.0,0.803,0.724,0.084,0.093,0.764,0.216,0.121,0.159,0.093,0.764
14,4,temporada_2024,857.0,0.745,0.846,0.098,0.091,1.622,0.695,0.590,0.095,0.091,1.622


In [11]:
rd["recall_por_tipo_L"].round(3)

,horizonte,bloque,tipo,semanas,recall,semanas_rango,recall_rango
4,2,temporada_2010,brote_nuevo,51.0,0.118,0,0.059
5,2,temporada_2010,en_curso,183.0,0.863,0,0.033
14,2,temporada_2015,brote_nuevo,60.0,0.067,0,0.000
15,2,temporada_2015,en_curso,497.0,0.771,0,0.014
16,2,temporada_2016,brote_nuevo,53.0,0.132,0,0.000
17,2,temporada_2016,en_curso,342.0,0.804,0,0.003
18,2,temporada_2017,brote_nuevo,79.0,0.329,0,0.000
19,2,temporada_2017,en_curso,460.0,0.815,0,0.004
26,2,temporada_2022,brote_nuevo,49.0,0.122,0,0.041
27,2,temporada_2022,en_curso,382.0,0.770,0,0.005


In [12]:
rd["antelacion_L_semilla_17"].round(3)

,horizonte,bloque,alerta,episodios,avisados_2_o_mas,ic95_inferior,ic95_superior
2,2,temporada_2010,alerta_050,30,0.033,0.000,0.100
7,2,temporada_2015,alerta_050,34,0.059,0.000,0.147
8,2,temporada_2016,alerta_050,34,0.206,0.088,0.353
9,2,temporada_2017,alerta_050,56,0.446,0.321,0.571
13,2,temporada_2022,alerta_050,38,0.132,0.026,0.237
14,2,temporada_2023,alerta_050,57,0.193,0.105,0.298
15,2,temporada_2024,alerta_050,23,0.435,0.217,0.652
16,2,seleccion_2022_2023,alerta_050,95,0.168,0.095,0.242
19,3,temporada_2010,alerta_050,30,0.067,0.000,0.167
24,3,temporada_2015,alerta_050,34,0.029,0.000,0.088


In [13]:
rd["conteo_por_fase_L"][["horizonte", "bloque", "fase", "n", "razon_mae", "razon_rmse", "mae_modelo",
    "rmse_modelo"]].round(3)

,horizonte,bloque,fase,n,razon_mae,razon_rmse,mae_modelo,rmse_modelo
12,2,temporada_2010,valle,2340.0,0.986,0.885,0.343,1.965
13,2,temporada_2010,subida,390.0,0.938,0.928,2.595,13.721
14,2,temporada_2010,pico,195.0,1.060,1.065,9.261,39.894
15,2,temporada_2010,bajada,455.0,0.823,0.659,3.695,13.108
32,2,temporada_2015,valle,1950.0,0.921,0.925,0.589,2.921
33,2,temporada_2015,subida,390.0,1.036,0.964,2.905,11.271
34,2,temporada_2015,pico,650.0,1.139,1.221,9.460,26.950
35,2,temporada_2015,bajada,455.0,0.501,0.467,2.762,7.572
36,2,temporada_2016,valle,780.0,1.092,0.844,0.552,1.314
37,2,temporada_2016,subida,1235.0,1.256,1.354,1.749,5.478


In [14]:
rd["calibradores_L_semilla_17"]

,horizonte,bloque,corte,metodo,filas_calibracion,positivos_calibracion,etiquetas_no_cerradas,ultima_etiqueta_usada
0,2,temporada_2008,2007-08-18,platt,0,0,0,NaT
1,2,temporada_2009,2008-08-16,platt,3315,83,65,2008-08-16
2,2,temporada_2010,2009-08-22,platt,6760,258,65,2009-08-22
3,2,temporada_2011,2010-08-21,platt,10140,491,65,2010-08-21
4,2,temporada_2012,2011-08-20,platt,13520,506,65,2011-08-20
5,2,temporada_2013,2012-08-18,platt,16900,566,65,2012-08-18
6,2,temporada_2014,2013-08-17,platt,20280,655,65,2013-08-17
7,2,temporada_2015,2014-08-16,platt,23660,709,65,2014-08-16
8,2,temporada_2016,2015-08-22,platt,27105,1258,65,2015-08-22
9,2,temporada_2017,2016-08-20,platt,30485,1661,65,2016-08-20


## 5 · Salidas: métricas y MLflow

In [15]:
content = {"dispositivo": device, "guardas": g, "adopta_L": dec["adopta_L"],
           **{k: diag.to_json(v) for k, v in dec.items() if isinstance(v, pd.DataFrame)},
           **{k: diag.to_json(v) for k, v in rd.items()}}
# numpy scalars (np.bool_, np.int64) to native types before json.dumps.
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
cell = lambda t, b, h, c: float(t[t.bloque.eq(b) & t.horizonte.eq(h)].iloc[0][c])
key = {f"{m}_{v}.{b[-4:]}.h{h}": cell(t, b, h, m)
       for v, t in (("L", dec["medianas_L"]), ("R", dec["medianas_R"]))
       for b in ("temporada_2022", "temporada_2023") for h in (2, 3, 4) for m in ("f1_050", "razon_mae")}
tracking.log_study(
    "mejora_modelo_fase1a_historia", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "1a", "preregistro": "6a29a8e", "enmienda_G1": "81e72a4",
                       "veredicto": "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in dec["veredicto"].itertuples()),
                       "adopta_L": str(dec["adopta_L"]), **{f"xgb.{k}": v for k, v in device.items()}},
            "metricas": key},
    artifacts=[METRICS], tables={"veredicto": dec["veredicto"], "antelacion": dec["antelacion"]},
    description="Fase 1a del estudio 06: V1 entrenada desde 2005 frente a desde 2017.")

C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio 8372af480d9646529005126b404573e2 en 'denguekay/mejora_modelo_fase1a_historia' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'8372af480d9646529005126b404573e2'